In [1]:
from google.colab import files
uploaded = files.upload()

Saving sales_final.parquet to sales_final.parquet


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

df = spark.read.parquet("/content/sales_final.parquet")
print("Nevalytų duomenų irašų kiekis:", df.count())

Nevalytų duomenų irašų kiekis: 520673


In [3]:
from pyspark.sql import functions as F, types as T

# 1. Nuskaitomas duomenų rinkinys
df = spark.read.parquet("/content/sales_final.parquet")

print("=== SCHEMA ===")
df.printSchema()

initial_row_count = df.count()
print(f"Pradinis įrašų kiekis: {initial_row_count}")

# 2. Referencinės taisyklės
allowed_segments = ["Consumer", "Corporate", "Home Office"]
allowed_payment_methods = ["Card", "Bank Transfer", "Cash", "PayPal"]
allowed_shipping_modes = ["Same Day", "First Class", "Second Class", "Standard Class"]
allowed_regions = ["East", "West", "North", "South", "Central"]

city_reference = {
    "Vilnius":      {"region": "East",    "lat": 54.6872, "lon": 25.2797},
    "Kaunas":       {"region": "Central", "lat": 54.8985, "lon": 23.9036},
    "Klaipėda":     {"region": "West",    "lat": 55.7033, "lon": 21.1443},
    "Šiauliai":     {"region": "North",   "lat": 55.9349, "lon": 23.3137},
    "Panevėžys":    {"region": "North",   "lat": 55.7348, "lon": 24.3575},
    "Alytus":       {"region": "South",   "lat": 54.3964, "lon": 24.0414},
    "Marijampolė":  {"region": "South",   "lat": 54.5599, "lon": 23.3541},
    "Jonava":       {"region": "Central", "lat": 55.0733, "lon": 24.2801},
    "Utena":        {"region": "East",    "lat": 55.4970, "lon": 25.5990},
    "Elektrėnai":   {"region": "Central", "lat": 54.7847, "lon": 24.6637},
    "Telšiai":      {"region": "West",    "lat": 55.9869, "lon": 22.2478},
    "Birštonas":    {"region": "South",   "lat": 54.6054, "lon": 24.0296},
    "Tauragė":      {"region": "West",    "lat": 55.2526, "lon": 22.2891},
    "Akmenė":       {"region": "North",   "lat": 56.2460, "lon": 22.7471},
    "Mažeikiai":    {"region": "North",   "lat": 56.3067, "lon": 22.3339},
    "Druskininkai": {"region": "South",   "lat": 54.0150, "lon": 23.9870},
    "Palanga":      {"region": "West",    "lat": 55.9167, "lon": 21.0667},
    "Visaginas":    {"region": "East",    "lat": 55.6000, "lon": 26.4167},
}

issues = []

def add_issue(issue_name, count_value, note):
    issues.append((issue_name, int(count_value), note))

# 3. NULL ir tuščių reikšmių suvestinė
null_exprs = []
for c, t in df.dtypes:
    if t == "string":
        null_exprs.append(
            F.sum(
                F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)
            ).alias(c)
        )
    else:
        null_exprs.append(
            F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        )

null_summary = df.select(null_exprs).collect()[0].asDict()

for col_name, bad_count in null_summary.items():
    if bad_count > 0:
        add_issue(
            f"NULL arba tuščios reikšmės stulpelyje {col_name}",
            bad_count,
            "Reikia atskirai įvertinti, ar reikšmės taisomos, ar šalinamos."
        )

# 4. Pilni dublikatai
duplicate_count = initial_row_count - df.dropDuplicates().count()
if duplicate_count > 0:
    add_issue(
        "Pilni eilučių dublikatai",
        duplicate_count,
        "Tai pilnai pasikartojančios eilutės, kandidatai į šalinimą po patikros."
    )

# 5. ID formatų tikrinimas
order_id_invalid = df.filter(
    F.col("Order_ID").isNull() |
    (~F.upper(F.trim(F.col("Order_ID"))).rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$"))
).count()
if order_id_invalid > 0:
    add_issue(
        "Netinkamas Order_ID formatas",
        order_id_invalid,
        "Taisyklė: AA-XXXX-XXXXXX."
    )

customer_id_invalid = df.filter(
    F.col("Customer_ID").isNull() |
    (~F.upper(F.trim(F.col("Customer_ID"))).rlike(r"^C-[0-9]{6}$"))
).count()
if customer_id_invalid > 0:
    add_issue(
        "Netinkamas Customer_ID formatas",
        customer_id_invalid,
        "Taisyklė: C-XXXXXX."
    )

# 6. Leistinų reikšmių tikrinimas
segment_invalid = df.filter(
    F.col("Segment").isNull() |
    (~F.trim(F.col("Segment")).isin(allowed_segments))
).count()
if segment_invalid > 0:
    add_issue(
        "Neleistinos Segment reikšmės",
        segment_invalid,
        "Leistinos reikšmės: Consumer, Corporate, Home Office."
    )

payment_invalid = df.filter(
    F.col("Payment_Method").isNull() |
    (~F.trim(F.col("Payment_Method")).isin(allowed_payment_methods))
).count()
if payment_invalid > 0:
    add_issue(
        "Neleistinos Payment_Method reikšmės",
        payment_invalid,
        "Leistinos reikšmės: Card, Bank Transfer, Cash, PayPal."
    )

shipping_invalid = df.filter(
    F.col("Shipping_Mode").isNull() |
    (~F.trim(F.col("Shipping_Mode")).isin(allowed_shipping_modes))
).count()
if shipping_invalid > 0:
    add_issue(
        "Neleistinos Shipping_Mode reikšmės",
        shipping_invalid,
        "Leistinos reikšmės: Same Day, First Class, Second Class, Standard Class."
    )

origin_region_invalid = df.filter(
    F.col("Origin_Region").isNull() |
    (~F.trim(F.col("Origin_Region")).isin(allowed_regions))
).count()
if origin_region_invalid > 0:
    add_issue(
        "Neleistinos Origin_Region reikšmės",
        origin_region_invalid,
        "Leistinos reikšmės: East, West, North, South, Central."
    )

destination_region_invalid = df.filter(
    F.col("Destination_Region").isNull() |
    (~F.trim(F.col("Destination_Region")).isin(allowed_regions))
).count()
if destination_region_invalid > 0:
    add_issue(
        "Neleistinos Destination_Region reikšmės",
        destination_region_invalid,
        "Leistinos reikšmės: East, West, North, South, Central."
    )

# 7. Datų tikrinimas
df_dates = (
    df.withColumn("Order_Date_ts", F.to_timestamp("Order_Date", "yyyy-MM-dd"))
      .withColumn("Ship_Date_ts", F.to_timestamp("Ship_Date", "yyyy-MM-dd"))
)

bad_order_date_parse = df_dates.filter(
    F.col("Order_Date").isNotNull() & F.col("Order_Date_ts").isNull()
).count()
if bad_order_date_parse > 0:
    add_issue(
        "Blogai išparsintos Order_Date reikšmės",
        bad_order_date_parse,
        "Datos neatitinka laukiamo formato."
    )

bad_ship_date_parse = df_dates.filter(
    F.col("Ship_Date").isNotNull() & F.col("Ship_Date_ts").isNull()
).count()
if bad_ship_date_parse > 0:
    add_issue(
        "Blogai išparsintos Ship_Date reikšmės",
        bad_ship_date_parse,
        "Datos neatitinka laukiamo formato."
    )

order_date_out_of_range = df_dates.filter(
    F.col("Order_Date_ts").isNotNull() &
    (
        (F.col("Order_Date_ts") < F.to_timestamp(F.lit("2025-01-01 00:00:00"))) |
        (F.col("Order_Date_ts") > F.to_timestamp(F.lit("2026-12-31 23:59:59")))
    )
).count()
if order_date_out_of_range > 0:
    add_issue(
        "Order_Date už leistino intervalo ribų",
        order_date_out_of_range,
        "Pagal aprašą data turi būti tarp 2025-01-01 ir 2026-12-31."
    )

ship_date_before_order = df_dates.filter(
    F.col("Ship_Date_ts").isNotNull() &
    F.col("Order_Date_ts").isNotNull() &
    (F.col("Ship_Date_ts") < F.col("Order_Date_ts"))
).count()
if ship_date_before_order > 0:
    add_issue(
        "Ship_Date ankstesnė už Order_Date",
        ship_date_before_order,
        "Pristatymo data negali būti ankstesnė už užsakymo datą."
    )

# 8. Tekstinių skaitinių laukų tikrinimas
gross_sales_dirty = df.filter(
    F.col("Gross_Sales").isNotNull() &
    (~F.trim(F.col("Gross_Sales")).rlike(r"^[0-9]+(\.[0-9]+)?$"))
).count()
if gross_sales_dirty > 0:
    add_issue(
        "Gross_Sales turi simbolių ar netinkamų ženklų",
        gross_sales_dirty,
        "Galimi valiutos simboliai, tekstas ar kiti neleistini ženklai."
    )

net_sales_dirty = df.filter(
    F.col("Net_Sales").isNotNull() &
    (~F.trim(F.col("Net_Sales")).rlike(r"^[0-9]+(\.[0-9]+)?$"))
).count()
if net_sales_dirty > 0:
    add_issue(
        "Net_Sales turi simbolių ar raidžių vietoje skaičių",
        net_sales_dirty,
        "Galimi O/I/S ir panašūs simboliai vietoje skaitmenų."
    )

# 9. Geografinių reikšmių tikrinimas
city_rows = []
for city, vals in city_reference.items():
    city_rows.append((city, vals["region"], vals["lat"], vals["lon"]))

city_schema = T.StructType([
    T.StructField("City", T.StringType(), False),
    T.StructField("Expected_Region", T.StringType(), False),
    T.StructField("Expected_Latitude", T.DoubleType(), False),
    T.StructField("Expected_Longitude", T.DoubleType(), False),
])

city_ref_df = spark.createDataFrame(city_rows, schema=city_schema)

origin_check = (
    df.join(city_ref_df, df["Origin_City"] == city_ref_df["City"], "left")
      .withColumn(
          "Origin_City_Not_In_List",
          F.when(F.col("City").isNull() & F.col("Origin_City").isNotNull(), 1).otherwise(0)
      )
      .withColumn(
          "Origin_Region_Mismatch",
          F.when(
              F.col("City").isNotNull() &
              (F.col("Origin_Region") != F.col("Expected_Region")),
              1
          ).otherwise(0)
      )
      .withColumn(
          "Origin_Lat_Mismatch",
          F.when(
              F.col("City").isNotNull() &
              (F.round(F.col("Origin_Latitude"), 4) != F.round(F.col("Expected_Latitude"), 4)),
              1
          ).otherwise(0)
      )
      .withColumn(
          "Origin_Lon_Mismatch",
          F.when(
              F.col("City").isNotNull() &
              (F.round(F.col("Origin_Longitude"), 4) != F.round(F.col("Expected_Longitude"), 4)),
              1
          ).otherwise(0)
      )
)

origin_city_not_in_list = origin_check.agg(F.sum("Origin_City_Not_In_List")).collect()[0][0]
origin_region_mismatch = origin_check.agg(F.sum("Origin_Region_Mismatch")).collect()[0][0]
origin_lat_mismatch = origin_check.agg(F.sum("Origin_Lat_Mismatch")).collect()[0][0]
origin_lon_mismatch = origin_check.agg(F.sum("Origin_Lon_Mismatch")).collect()[0][0]

if origin_city_not_in_list > 0:
    add_issue(
        "Origin_City nėra referenciniame miestų sąraše",
        origin_city_not_in_list,
        "Miestas neatitinka pateikto leistinų miestų sąrašo."
    )
if origin_region_mismatch > 0:
    add_issue(
        "Origin_Region neatitinka Origin_City",
        origin_region_mismatch,
        "Regionas nesutampa su miesto reference mapping."
    )
if origin_lat_mismatch > 0:
    add_issue(
        "Origin_Latitude neatitinka Origin_City",
        origin_lat_mismatch,
        "Platuma nesutampa su reference mapping."
    )
if origin_lon_mismatch > 0:
    add_issue(
        "Origin_Longitude neatitinka Origin_City",
        origin_lon_mismatch,
        "Ilguma nesutampa su reference mapping."
    )

destination_check = (
    df.join(city_ref_df, df["Destination_City"] == city_ref_df["City"], "left")
      .withColumn(
          "Destination_City_Not_In_List",
          F.when(F.col("City").isNull() & F.col("Destination_City").isNotNull(), 1).otherwise(0)
      )
      .withColumn(
          "Destination_Region_Mismatch",
          F.when(
              F.col("City").isNotNull() &
              (F.col("Destination_Region") != F.col("Expected_Region")),
              1
          ).otherwise(0)
      )
      .withColumn(
          "Destination_Lat_Mismatch",
          F.when(
              F.col("City").isNotNull() &
              (F.round(F.col("Destination_Latitude"), 4) != F.round(F.col("Expected_Latitude"), 4)),
              1
          ).otherwise(0)
      )
      .withColumn(
          "Destination_Lon_Mismatch",
          F.when(
              F.col("City").isNotNull() &
              (F.round(F.col("Destination_Longitude"), 4) != F.round(F.col("Expected_Longitude"), 4)),
              1
          ).otherwise(0)
      )
)

destination_city_not_in_list = destination_check.agg(F.sum("Destination_City_Not_In_List")).collect()[0][0]
destination_region_mismatch = destination_check.agg(F.sum("Destination_Region_Mismatch")).collect()[0][0]
destination_lat_mismatch = destination_check.agg(F.sum("Destination_Lat_Mismatch")).collect()[0][0]
destination_lon_mismatch = destination_check.agg(F.sum("Destination_Lon_Mismatch")).collect()[0][0]

if destination_city_not_in_list > 0:
    add_issue(
        "Destination_City nėra referenciniame miestų sąraše",
        destination_city_not_in_list,
        "Miestas neatitinka pateikto leistinų miestų sąrašo."
    )
if destination_region_mismatch > 0:
    add_issue(
        "Destination_Region neatitinka Destination_City",
        destination_region_mismatch,
        "Regionas nesutampa su miesto reference mapping."
    )
if destination_lat_mismatch > 0:
    add_issue(
        "Destination_Latitude neatitinka Destination_City",
        destination_lat_mismatch,
        "Platuma nesutampa su reference mapping."
    )
if destination_lon_mismatch > 0:
    add_issue(
        "Destination_Longitude neatitinka Destination_City",
        destination_lon_mismatch,
        "Ilguma nesutampa su reference mapping."
    )

# 10. Paprastos verslo logikos patikros
quantity_invalid = df.filter(
    F.col("Quantity").isNull() |
    (F.col("Quantity") < 1) |
    (F.col("Quantity") > 10)
).count()
if quantity_invalid > 0:
    add_issue(
        "Quantity už leistinų ribų",
        quantity_invalid,
        "Pagal aprašą Quantity turi būti tarp 1 ir 10."
    )

discount_invalid = df.filter(
    F.col("Discount").isNull() |
    (F.col("Discount") < 0.0) |
    (F.col("Discount") > 0.3)
).count()
if discount_invalid > 0:
    add_issue(
        "Discount už leistinų ribų",
        discount_invalid,
        "Pagal aprašą Discount turi būti tarp 0.0 ir 0.3."
    )

distance_invalid = df.filter(
    F.col("Distance_km").isNull() | (F.col("Distance_km") <= 0)
).count()
if distance_invalid > 0:
    add_issue(
        "Distance_km nelogiška arba neveiksni",
        distance_invalid,
        "Atstumas turi būti teigiamas."
    )

weight_invalid = df.filter(
    F.col("Weight_kg").isNull() | (F.col("Weight_kg") <= 0)
).count()
if weight_invalid > 0:
    add_issue(
        "Weight_kg nelogiška arba neveiksni",
        weight_invalid,
        "Svoris turi būti teigiamas."
    )

cogs_invalid = df.filter(
    F.col("COGS").isNull() | (F.col("COGS") <= 0)
).count()
if cogs_invalid > 0:
    add_issue(
        "COGS nelogiška arba neveiksni",
        cogs_invalid,
        "COGS turi būti teigiamas."
    )

# 11. Oficialus problemų sąrašas
issues_df = spark.createDataFrame(
    issues,
    schema=["Problema", "Neatitikimų_kiekis", "Pastaba"]
).orderBy(F.desc("Neatitikimų_kiekis"))

print("=== OFICIALUS PROBLEMŲ SĄRAŠAS ===")
issues_df.show(200, truncate=False)

print("=== NULL / TUŠČIŲ REIKŠMIŲ SUVESTINĖ ===")
for col_name, bad_count in null_summary.items():
    if bad_count > 0:
        print(f"{col_name}: {bad_count}")

print(f"Pilnų dublikatų kiekis: {duplicate_count}")

=== SCHEMA ===
root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship_Date: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Shipping_Mode: string (nullable = true)
 |-- Origin_City: string (nullable = true)
 |-- Origin_Region: string (nullable = true)
 |-- Origin_Latitude: double (nullable = true)
 |-- Origin_Longitude: double (nullable = true)
 |-- Destination_City: string (nullable = true)
 |-- Destination_Region: string (nullable = true)
 |-- Destination_Latitude: double (nullable = true)
 |-- Destination_Longitude: double (nullable = true)
 |-- Distance_km: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Weight_kg: double (nullable = true)
 |-- COGS: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Gross_Sales: string (nullable = true)
 |-- Net_Sales: string (nullable = true)
 |-- Profit: dou

In [4]:
from pyspark.sql import functions as F

# 1. Kiek blogų reikšmių buvo prieš tvarkymą
gross_sales_dirty_before = df.filter(
    F.col("Gross_Sales").isNotNull() &
    (~F.trim(F.col("Gross_Sales")).rlike(r"^[0-9]+(\.[0-9]+)?$"))
).count()

net_sales_dirty_before = df.filter(
    F.col("Net_Sales").isNotNull() &
    (~F.trim(F.col("Net_Sales")).rlike(r"^[0-9]+(\.[0-9]+)?$"))
).count()

print("PRIEŠ TVARKYMĄ")
print("Gross_Sales dirty:", gross_sales_dirty_before)
print("Net_Sales dirty:", net_sales_dirty_before)

# 2. Taisymas
df = df.withColumn(
    "Gross_Sales",
    F.trim(F.col("Gross_Sales"))
)

df = df.withColumn(
    "Gross_Sales",
    F.regexp_replace(F.col("Gross_Sales"), r"(?i)\s*eur\s*", "")
)

df = df.withColumn(
    "Gross_Sales",
    F.regexp_replace(F.col("Gross_Sales"), r"[€$]", "")
)

df = df.withColumn(
    "Gross_Sales",
    F.trim(F.col("Gross_Sales"))
)

df = df.withColumn(
    "Net_Sales",
    F.upper(F.trim(F.col("Net_Sales")))
)

df = df.withColumn(
    "Net_Sales",
    F.regexp_replace(F.col("Net_Sales"), r"O", "0")
)

df = df.withColumn(
    "Net_Sales",
    F.regexp_replace(F.col("Net_Sales"), r"I", "1")
)

df = df.withColumn(
    "Net_Sales",
    F.regexp_replace(F.col("Net_Sales"), r"S", "5")
)

df = df.withColumn(
    "Net_Sales",
    F.trim(F.col("Net_Sales"))
)

# 3. Po taisymo vėl tikrinam formatą
gross_sales_dirty_after = df.filter(
    F.col("Gross_Sales").isNotNull() &
    (~F.trim(F.col("Gross_Sales")).rlike(r"^[0-9]+(\.[0-9]+)?$"))
).count()

net_sales_dirty_after = df.filter(
    F.col("Net_Sales").isNotNull() &
    (~F.trim(F.col("Net_Sales")).rlike(r"^[0-9]+(\.[0-9]+)?$"))
).count()

print("PO TVARKYMO")
print("Gross_Sales dirty:", gross_sales_dirty_after)
print("Net_Sales dirty:", net_sales_dirty_after)

# 4. Pasitikrinimui parodyti keletą pavyzdžių
print("Gross_Sales pavyzdžiai po tvarkymo:")
df.select("Gross_Sales").filter(F.col("Gross_Sales").isNotNull()).show(10, truncate=False)

print("Net_Sales pavyzdžiai po tvarkymo:")
df.select("Net_Sales").filter(F.col("Net_Sales").isNotNull()).show(10, truncate=False)

PRIEŠ TVARKYMĄ
Gross_Sales dirty: 2507
Net_Sales dirty: 1821
PO TVARKYMO
Gross_Sales dirty: 0
Net_Sales dirty: 0
Gross_Sales pavyzdžiai po tvarkymo:
+------------------+
|Gross_Sales       |
+------------------+
|64.206797446571   |
|46.87311289483167 |
|160.56871672691184|
|290.60998009822555|
|458.9823277864826 |
|241.33077455171502|
|249.60973492016637|
|87.71125844124586 |
|83.42398695864344 |
|139.79219658406876|
+------------------+
only showing top 10 rows
Net_Sales pavyzdžiai po tvarkymo:
+------------------+
|Net_Sales         |
+------------------+
|64.206797446571   |
|46.87311289483167 |
|118.34416390531882|
|222.2825514814393 |
|426.73733717939035|
|170.05749275893947|
|229.86828440824212|
|87.71125844124586 |
|83.42398695864344 |
|139.79219658406876|
+------------------+
only showing top 10 rows


In [5]:
from pyspark.sql import functions as F

# 1. Kiek null buvo prieš cast
gross_null_before_cast = df.filter(F.col("Gross_Sales").isNull()).count()
net_null_before_cast = df.filter(F.col("Net_Sales").isNull()).count()

print("PRIEŠ CAST")
print("Gross_Sales null:", gross_null_before_cast)
print("Net_Sales null:", net_null_before_cast)

# 2. Cast į double
df = df.withColumn("Gross_Sales", F.col("Gross_Sales").cast("double"))
df = df.withColumn("Net_Sales", F.col("Net_Sales").cast("double"))

# 3. Kiek null po cast
gross_null_after_cast = df.filter(F.col("Gross_Sales").isNull()).count()
net_null_after_cast = df.filter(F.col("Net_Sales").isNull()).count()

print("PO CAST")
print("Gross_Sales null:", gross_null_after_cast)
print("Net_Sales null:", net_null_after_cast)

# 4. Ar atsirado naujų null dėl nesėkmingo cast
gross_new_nulls_from_cast = gross_null_after_cast - gross_null_before_cast
net_new_nulls_from_cast = net_null_after_cast - net_null_before_cast

print("NAUJAI ATSIRADĘ NULL PO CAST")
print("Gross_Sales new nulls:", gross_new_nulls_from_cast)
print("Net_Sales new nulls:", net_new_nulls_from_cast)

# 5. Finansinės logikos tikrinimas
net_gt_gross_count = df.filter(
    F.col("Net_Sales").isNotNull() &
    F.col("Gross_Sales").isNotNull() &
    (F.col("Net_Sales") > F.col("Gross_Sales"))
).count()

profit_mismatch_count = df.filter(
    F.col("Profit").isNotNull() &
    F.col("Net_Sales").isNotNull() &
    F.col("COGS").isNotNull() &
    (F.abs(F.col("Profit") - (F.col("Net_Sales") - F.col("COGS"))) > 0.01)
).count()

print("FINANSINĖ LOGIKA")
print("Net_Sales > Gross_Sales:", net_gt_gross_count)
print("Profit != Net_Sales - COGS:", profit_mismatch_count)

# 6. Papildomai pasitikrinimui parodyti keletą blogų eilučių, jei tokių yra
print("Pavyzdžiai, kur Net_Sales > Gross_Sales:")
df.filter(
    F.col("Net_Sales").isNotNull() &
    F.col("Gross_Sales").isNotNull() &
    (F.col("Net_Sales") > F.col("Gross_Sales"))
).select("Order_ID", "Gross_Sales", "Net_Sales", "COGS", "Profit").show(10, truncate=False)

print("Pavyzdžiai, kur Profit != Net_Sales - COGS:")
df.filter(
    F.col("Profit").isNotNull() &
    F.col("Net_Sales").isNotNull() &
    F.col("COGS").isNotNull() &
    (F.abs(F.col("Profit") - (F.col("Net_Sales") - F.col("COGS"))) > 0.01)
).select("Order_ID", "Gross_Sales", "Net_Sales", "COGS", "Profit").show(10, truncate=False)

PRIEŠ CAST
Gross_Sales null: 0
Net_Sales null: 0
PO CAST
Gross_Sales null: 0
Net_Sales null: 0
NAUJAI ATSIRADĘ NULL PO CAST
Gross_Sales new nulls: 0
Net_Sales new nulls: 0
FINANSINĖ LOGIKA
Net_Sales > Gross_Sales: 0
Profit != Net_Sales - COGS: 0
Pavyzdžiai, kur Net_Sales > Gross_Sales:
+--------+-----------+---------+----+------+
|Order_ID|Gross_Sales|Net_Sales|COGS|Profit|
+--------+-----------+---------+----+------+
+--------+-----------+---------+----+------+

Pavyzdžiai, kur Profit != Net_Sales - COGS:
+--------+-----------+---------+----+------+
|Order_ID|Gross_Sales|Net_Sales|COGS|Profit|
+--------+-----------+---------+----+------+
+--------+-----------+---------+----+------+



In [6]:
from pyspark.sql import functions as F

# 1. Bendras eilučių skaičius prieš šalinimą
row_count_before = df.count()

# 2. Kiek yra pilnų dublikatų
duplicate_count_before = row_count_before - df.dropDuplicates().count()

print("PRIEŠ ŠALINIMĄ")
print("Row count:", row_count_before)
print("Full duplicate rows:", duplicate_count_before)

# 3. Pašalinami pilni dublikatai
df = df.dropDuplicates()

# 4. Patikra po šalinimo
row_count_after = df.count()
duplicate_count_after = row_count_after - df.dropDuplicates().count()

print("PO ŠALINIMO")
print("Row count:", row_count_after)
print("Full duplicate rows:", duplicate_count_after)

# 5. Kiek eilučių realiai pašalinta
removed_rows = row_count_before - row_count_after

print("PAŠALINTA EILUČIŲ:", removed_rows)

PRIEŠ ŠALINIMĄ
Row count: 520673
Full duplicate rows: 4228
PO ŠALINIMO
Row count: 516445
Full duplicate rows: 0
PAŠALINTA EILUČIŲ: 4228


In [7]:
from pyspark.sql import functions as F

# 1. Kiek NULL buvo prieš konvertavimą
order_date_null_before = df.filter(F.col("Order_Date").isNull()).count()
ship_date_null_before = df.filter(F.col("Ship_Date").isNull()).count()

print("PRIEŠ KONVERTAVIMĄ")
print("Order_Date null:", order_date_null_before)
print("Ship_Date null:", ship_date_null_before)

# 2. Konvertuojam į timestamp
df = df.withColumn("Order_Date", F.to_timestamp(F.col("Order_Date"), "yyyy-MM-dd"))
df = df.withColumn("Ship_Date", F.to_timestamp(F.col("Ship_Date"), "yyyy-MM-dd"))

# 3. Kiek NULL po konvertavimo
order_date_null_after = df.filter(F.col("Order_Date").isNull()).count()
ship_date_null_after = df.filter(F.col("Ship_Date").isNull()).count()

print("PO KONVERTAVIMO")
print("Order_Date null:", order_date_null_after)
print("Ship_Date null:", ship_date_null_after)

# 4. Kiek naujų NULL atsirado dėl blogo formato
order_date_new_nulls = order_date_null_after - order_date_null_before
ship_date_new_nulls = ship_date_null_after - ship_date_null_before

print("NAUJAI ATSIRADĘ NULL")
print("Order_Date new nulls:", order_date_new_nulls)
print("Ship_Date new nulls:", ship_date_new_nulls)

# 5. Tikrinam datų logiką
ship_before_order_count = df.filter(
    F.col("Order_Date").isNotNull() &
    F.col("Ship_Date").isNotNull() &
    (F.col("Ship_Date") < F.col("Order_Date"))
).count()

print("DATŲ LOGIKA")
print("Ship_Date < Order_Date:", ship_before_order_count)

# 6. Jei reikia, parodom kelis blogus pavyzdžius
df.filter(
    F.col("Order_Date").isNotNull() &
    F.col("Ship_Date").isNotNull() &
    (F.col("Ship_Date") < F.col("Order_Date"))
).select(
    "Order_ID", "Order_Date", "Ship_Date", "Shipping_Mode"
).show(10, truncate=False)

# 7. Pasitikrinam schemą
df.printSchema()

PRIEŠ KONVERTAVIMĄ
Order_Date null: 0
Ship_Date null: 0
PO KONVERTAVIMO
Order_Date null: 0
Ship_Date null: 0
NAUJAI ATSIRADĘ NULL
Order_Date new nulls: 0
Ship_Date new nulls: 0
DATŲ LOGIKA
Ship_Date < Order_Date: 0
+--------+----------+---------+-------------+
|Order_ID|Order_Date|Ship_Date|Shipping_Mode|
+--------+----------+---------+-------------+
+--------+----------+---------+-------------+

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Order_Date: timestamp (nullable = true)
 |-- Ship_Date: timestamp (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Shipping_Mode: string (nullable = true)
 |-- Origin_City: string (nullable = true)
 |-- Origin_Region: string (nullable = true)
 |-- Origin_Latitude: double (nullable = true)
 |-- Origin_Longitude: double (nullable = true)
 |-- Destination_City: string (nullable = true)
 |-- Destination_Region: string (nullable = true)
 |-- Destina

In [8]:
from pyspark.sql import functions as F

# 1. Normalizuota pagalbinė versija tik analizei
order_id_profile_df = df.withColumn(
    "Order_ID_norm",
    F.upper(F.trim(F.col("Order_ID")))
)

# 2. Suklasifikuojam Order_ID problemas į tipus
order_id_profile_df = order_id_profile_df.withColumn(
    "Order_ID_issue_type",
    F.when(F.col("Order_ID").isNull(), "NULL")
     .when(F.col("Order_ID_norm").rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$"), "VALID")
     .when(~F.col("Order_ID_norm").contains("-"), "NO_HYPHENS")
     .when(F.length(F.col("Order_ID_norm")) < 14, "TOO_SHORT")
     .when(F.length(F.col("Order_ID_norm")) > 14, "TOO_LONG")
     .when(
         F.col("Order_ID_norm").rlike(r"^[0-9]{2}-[0-9]{4}-[0-9]{6}$"),
         "DIGITS_IN_PREFIX"
     )
     .when(
         F.col("Order_ID_norm").rlike(r"^[A-Z]{2}-[0-9]+-[0-9]+$"),
         "BAD_DIGIT_GROUP_LENGTH"
     )
     .when(
         F.col("Order_ID_norm").rlike(r"^[A-Z0-9]{2}-[A-Z0-9]{4}-[A-Z0-9]{6}$"),
         "ALPHANUMERIC_STRUCTURE_BROKEN"
     )
     .otherwise("OTHER")
)

# 3. Suvestinė pagal problemos tipą
order_id_issue_summary = order_id_profile_df.groupBy("Order_ID_issue_type").count().orderBy(F.desc("count"))

print("ORDER_ID PROBLEMŲ SUVESTINĖ")
order_id_issue_summary.show(50, truncate=False)

# 4. Parodom po kelis pavyzdžius iš kiekvienos blogos kategorijos
problem_types = [
    row["Order_ID_issue_type"]
    for row in order_id_issue_summary.collect()
    if row["Order_ID_issue_type"] != "VALID"
]

for issue_type in problem_types:
    print(f"\nPAVYZDŽIAI: {issue_type}")
    order_id_profile_df.filter(
        F.col("Order_ID_issue_type") == issue_type
    ).select(
        "Order_ID",
        "Order_ID_norm"
    ).show(10, truncate=False)

# 5. Papildomai suskaičiuojam, kiek iš invalid įrašų galima būtų pataisyti vien normalizavimu
order_id_fixable_by_trim_upper = df.filter(
    F.col("Order_ID").isNotNull() &
    (~F.col("Order_ID").rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$")) &
    (F.upper(F.trim(F.col("Order_ID"))).rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$"))
).count()

print("Pataisomi vien TRIM + UPPER:", order_id_fixable_by_trim_upper)

ORDER_ID PROBLEMŲ SUVESTINĖ
+-------------------+------+
|Order_ID_issue_type|count |
+-------------------+------+
|VALID              |513628|
|NULL               |705   |
|TOO_SHORT          |704   |
|DIGITS_IN_PREFIX   |704   |
|TOO_LONG           |704   |
+-------------------+------+


PAVYZDŽIAI: NULL
+--------+-------------+
|Order_ID|Order_ID_norm|
+--------+-------------+
|NULL    |NULL         |
|NULL    |NULL         |
|NULL    |NULL         |
|NULL    |NULL         |
|NULL    |NULL         |
|NULL    |NULL         |
|NULL    |NULL         |
|NULL    |NULL         |
|NULL    |NULL         |
|NULL    |NULL         |
+--------+-------------+
only showing top 10 rows

PAVYZDŽIAI: TOO_SHORT
+------------+-------------+
|Order_ID    |Order_ID_norm|
+------------+-------------+
|AB-123-12345|AB-123-12345 |
|AB-123-12345|AB-123-12345 |
|AB-123-12345|AB-123-12345 |
|AB-123-12345|AB-123-12345 |
|AB-123-12345|AB-123-12345 |
|AB-123-12345|AB-123-12345 |
|AB-123-12345|AB-123-12345 |
|AB-

In [9]:
from pyspark.sql import functions as F

# 1. Origin pusės neatitikimai pagal miestą ir regioną
origin_region_mismatch_df = df.filter(
    F.col("Origin_City").isNotNull() &
    F.col("Origin_Region").isNotNull()
).groupBy("Origin_City", "Origin_Region").count().orderBy(F.desc("count"))

print("ORIGIN CITY / REGION PASISKIRSTYMAS")
origin_region_mismatch_df.show(100, truncate=False)

# 2. Destination pusės neatitikimai pagal miestą ir regioną
destination_region_mismatch_df = df.filter(
    F.col("Destination_City").isNotNull() &
    F.col("Destination_Region").isNotNull()
).groupBy("Destination_City", "Destination_Region").count().orderBy(F.desc("count"))

print("DESTINATION CITY / REGION PASISKIRSTYMAS")
destination_region_mismatch_df.show(100, truncate=False)

# 3. Tik neatitikimų eilutės origin pusei pagal reference mapping
origin_reference_check = (
    df.join(city_ref_df, df["Origin_City"] == city_ref_df["City"], "left")
      .filter(
          F.col("City").isNotNull() &
          (F.col("Origin_Region") != F.col("Expected_Region"))
      )
      .groupBy("Origin_City", "Origin_Region", "Expected_Region")
      .count()
      .orderBy(F.desc("count"))
)

print("ORIGIN REGION NEATITIKIMAI PAGAL REFERENCE")
origin_reference_check.show(100, truncate=False)

# 4. Tik neatitikimų eilutės destination pusei pagal reference mapping
destination_reference_check = (
    df.join(city_ref_df, df["Destination_City"] == city_ref_df["City"], "left")
      .filter(
          F.col("City").isNotNull() &
          (F.col("Destination_Region") != F.col("Expected_Region"))
      )
      .groupBy("Destination_City", "Destination_Region", "Expected_Region")
      .count()
      .orderBy(F.desc("count"))
)

print("DESTINATION REGION NEATITIKIMAI PAGAL REFERENCE")
destination_reference_check.show(100, truncate=False)

# 5. Papildoma saugumo patikra: ar neatitikimų eilutėse koordinatės sutampa su reference
origin_coord_check = (
    df.join(city_ref_df, df["Origin_City"] == city_ref_df["City"], "left")
      .filter(
          F.col("City").isNotNull() &
          (F.col("Origin_Region") != F.col("Expected_Region"))
      )
      .withColumn(
          "lat_match",
          F.round(F.col("Origin_Latitude"), 4) == F.round(F.col("Expected_Latitude"), 4)
      )
      .withColumn(
          "lon_match",
          F.round(F.col("Origin_Longitude"), 4) == F.round(F.col("Expected_Longitude"), 4)
      )
      .groupBy("Origin_City", "Origin_Region", "Expected_Region", "lat_match", "lon_match")
      .count()
      .orderBy(F.desc("count"))
)

print("ORIGIN NEATITIKIMAI SU KOORDINAČIŲ PATIKRA")
origin_coord_check.show(100, truncate=False)

destination_coord_check = (
    df.join(city_ref_df, df["Destination_City"] == city_ref_df["City"], "left")
      .filter(
          F.col("City").isNotNull() &
          (F.col("Destination_Region") != F.col("Expected_Region"))
      )
      .withColumn(
          "lat_match",
          F.round(F.col("Destination_Latitude"), 4) == F.round(F.col("Expected_Latitude"), 4)
      )
      .withColumn(
          "lon_match",
          F.round(F.col("Destination_Longitude"), 4) == F.round(F.col("Expected_Longitude"), 4)
      )
      .groupBy("Destination_City", "Destination_Region", "Expected_Region", "lat_match", "lon_match")
      .count()
      .orderBy(F.desc("count"))
)

print("DESTINATION NEATITIKIMAI SU KOORDINAČIŲ PATIKRA")
destination_coord_check.show(100, truncate=False)

ORIGIN CITY / REGION PASISKIRSTYMAS
+------------+-------------+-----+
|Origin_City |Origin_Region|count|
+------------+-------------+-----+
|Visaginas   |East         |29009|
|Klaipėda    |West         |28913|
|Elektrėnai  |Central      |28911|
|Telšiai     |West         |28871|
|Alytus      |South        |28864|
|Birštonas   |Central      |28802|
|Druskininkai|South        |28788|
|Jonava      |Central      |28775|
|Panevėžys   |North        |28741|
|Mažeikiai   |North        |28670|
|Marijampolė |South        |28614|
|Palanga     |West         |28610|
|Akmenė      |North        |28575|
|Tauragė     |West         |28545|
|Šiauliai    |North        |28525|
|Vilnius     |East         |28462|
|Utena       |East         |28388|
|Kaunas      |Central      |28382|
+------------+-------------+-----+

DESTINATION CITY / REGION PASISKIRSTYMAS
+----------------+------------------+-----+
|Destination_City|Destination_Region|count|
+----------------+------------------+-----+
|Marijampolė     |So

In [10]:
from pyspark.sql import functions as F

# 1. Kiek neatitikimų prieš taisymą
origin_region_mismatch_before = df.filter(
    (F.col("Origin_City") == "Birštonas") &
    (F.round(F.col("Origin_Latitude"), 4) == 54.6054) &
    (F.round(F.col("Origin_Longitude"), 4) == 24.0296) &
    (F.col("Origin_Region") != "South")
).count()

destination_region_mismatch_before = df.filter(
    (F.col("Destination_City") == "Birštonas") &
    (F.round(F.col("Destination_Latitude"), 4) == 54.6054) &
    (F.round(F.col("Destination_Longitude"), 4) == 24.0296) &
    (F.col("Destination_Region") != "South")
).count()

print("PRIEŠ TAISYMĄ")
print("Origin Birštonas region mismatch:", origin_region_mismatch_before)
print("Destination Birštonas region mismatch:", destination_region_mismatch_before)

# 2. Origin_Region taisymas
df = df.withColumn(
    "Origin_Region",
    F.when(
        (F.col("Origin_City") == "Birštonas") &
        (F.round(F.col("Origin_Latitude"), 4) == 54.6054) &
        (F.round(F.col("Origin_Longitude"), 4) == 24.0296) &
        (F.col("Origin_Region") != "South"),
        F.lit("South")
    ).otherwise(F.col("Origin_Region"))
)

# 3. Destination_Region taisymas
df = df.withColumn(
    "Destination_Region",
    F.when(
        (F.col("Destination_City") == "Birštonas") &
        (F.round(F.col("Destination_Latitude"), 4) == 54.6054) &
        (F.round(F.col("Destination_Longitude"), 4) == 24.0296) &
        (F.col("Destination_Region") != "South"),
        F.lit("South")
    ).otherwise(F.col("Destination_Region"))
)

# 4. Kiek neatitikimų po taisymo
origin_region_mismatch_after = df.filter(
    (F.col("Origin_City") == "Birštonas") &
    (F.round(F.col("Origin_Latitude"), 4) == 54.6054) &
    (F.round(F.col("Origin_Longitude"), 4) == 24.0296) &
    (F.col("Origin_Region") != "South")
).count()

destination_region_mismatch_after = df.filter(
    (F.col("Destination_City") == "Birštonas") &
    (F.round(F.col("Destination_Latitude"), 4) == 54.6054) &
    (F.round(F.col("Destination_Longitude"), 4) == 24.0296) &
    (F.col("Destination_Region") != "South")
).count()

print("PO TAISYMO")
print("Origin Birštonas region mismatch:", origin_region_mismatch_after)
print("Destination Birštonas region mismatch:", destination_region_mismatch_after)

# 5. Kiek eilučių realiai pataisyta
total_fixed = (
    origin_region_mismatch_before - origin_region_mismatch_after +
    destination_region_mismatch_before - destination_region_mismatch_after
)

print("IŠ VISO PATAISYTA EILUČIŲ:", total_fixed)

PRIEŠ TAISYMĄ
Origin Birštonas region mismatch: 28802
Destination Birštonas region mismatch: 28726
PO TAISYMO
Origin Birštonas region mismatch: 0
Destination Birštonas region mismatch: 0
IŠ VISO PATAISYTA EILUČIŲ: 57528


In [11]:
from pyspark.sql import functions as F

# 1. Bendras eilučių skaičius po visų atliktų taisymų
final_row_count = df.count()
print("GALUTINIS ĮRAŠŲ KIEKIS:", final_row_count)

# 2. NULL / tuščių reikšmių suvestinė
null_exprs = []
for c, t in df.dtypes:
    if t == "string":
        null_exprs.append(
            F.sum(
                F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)
            ).alias(c)
        )
    else:
        null_exprs.append(
            F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        )

null_summary_after = df.select(null_exprs).collect()[0].asDict()

print("NULL / TUŠČIŲ REIKŠMIŲ SUVESTINĖ PO TVARKYMO")
for col_name, bad_count in null_summary_after.items():
    if bad_count > 0:
        print(f"{col_name}: {bad_count}")

# 3. Pilni dublikatai po tvarkymo
duplicate_count_after = final_row_count - df.dropDuplicates().count()
print("PILNŲ DUBLIKATŲ PO TVARKYMO:", duplicate_count_after)

# 4. Order_ID formatas po tvarkymo
order_id_invalid_after = df.filter(
    F.col("Order_ID").isNull() |
    (~F.upper(F.trim(F.col("Order_ID"))).rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$"))
).count()

print("NETINKAMAS Order_ID FORMATAS PO TVARKYMO:", order_id_invalid_after)

# 5. Gross_Sales ir Net_Sales validumas po tvarkymo
gross_sales_dirty_after = df.filter(
    F.col("Gross_Sales").isNotNull() &
    (F.col("Gross_Sales").cast("double").isNull())
).count()

net_sales_dirty_after = df.filter(
    F.col("Net_Sales").isNotNull() &
    (F.col("Net_Sales").cast("double").isNull())
).count()

print("Gross_Sales dirty po tvarkymo:", gross_sales_dirty_after)
print("Net_Sales dirty po tvarkymo:", net_sales_dirty_after)

# 6. Datų logika po tvarkymo
ship_before_order_after = df.filter(
    F.col("Order_Date").isNotNull() &
    F.col("Ship_Date").isNotNull() &
    (F.col("Ship_Date") < F.col("Order_Date"))
).count()

print("Ship_Date < Order_Date po tvarkymo:", ship_before_order_after)

# 7. Regionų logika po tvarkymo (Birštono atvejis jau turėtų būti sutvarkytas)
origin_birstonas_mismatch_after = df.filter(
    (F.col("Origin_City") == "Birštonas") &
    (F.round(F.col("Origin_Latitude"), 4) == 54.6054) &
    (F.round(F.col("Origin_Longitude"), 4) == 24.0296) &
    (F.col("Origin_Region") != "South")
).count()

destination_birstonas_mismatch_after = df.filter(
    (F.col("Destination_City") == "Birštonas") &
    (F.round(F.col("Destination_Latitude"), 4) == 54.6054) &
    (F.round(F.col("Destination_Longitude"), 4) == 24.0296) &
    (F.col("Destination_Region") != "South")
).count()

print("Origin Birštonas region mismatch po tvarkymo:", origin_birstonas_mismatch_after)
print("Destination Birštonas region mismatch po tvarkymo:", destination_birstonas_mismatch_after)

# 8. Finansinė logika po tvarkymo
net_gt_gross_after = df.filter(
    F.col("Net_Sales").isNotNull() &
    F.col("Gross_Sales").isNotNull() &
    (F.col("Net_Sales") > F.col("Gross_Sales"))
).count()

profit_mismatch_after = df.filter(
    F.col("Profit").isNotNull() &
    F.col("Net_Sales").isNotNull() &
    F.col("COGS").isNotNull() &
    (F.abs(F.col("Profit") - (F.col("Net_Sales") - F.col("COGS"))) > 0.01)
).count()

print("Net_Sales > Gross_Sales po tvarkymo:", net_gt_gross_after)
print("Profit != Net_Sales - COGS po tvarkymo:", profit_mismatch_after)

GALUTINIS ĮRAŠŲ KIEKIS: 516445
NULL / TUŠČIŲ REIKŠMIŲ SUVESTINĖ PO TVARKYMO
Order_ID: 705
PILNŲ DUBLIKATŲ PO TVARKYMO: 0
NETINKAMAS Order_ID FORMATAS PO TVARKYMO: 2817
Gross_Sales dirty po tvarkymo: 0
Net_Sales dirty po tvarkymo: 0
Ship_Date < Order_Date po tvarkymo: 0
Origin Birštonas region mismatch po tvarkymo: 0
Destination Birštonas region mismatch po tvarkymo: 0
Net_Sales > Gross_Sales po tvarkymo: 0
Profit != Net_Sales - COGS po tvarkymo: 0


In [12]:
from pyspark.sql import functions as F

# 1. Normalizuota versija analizei
order_id_check_df = df.withColumn(
    "Order_ID_norm",
    F.upper(F.trim(F.col("Order_ID")))
)

# 2. Klasifikacija
order_id_check_df = order_id_check_df.withColumn(
    "Order_ID_issue_type",
    F.when(F.col("Order_ID").isNull(), "NULL")
     .when(F.col("Order_ID_norm").rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$"), "VALID")
     .when(F.length(F.col("Order_ID_norm")) < 14, "TOO_SHORT")
     .when(F.length(F.col("Order_ID_norm")) > 14, "TOO_LONG")
     .when(F.col("Order_ID_norm").rlike(r"^[0-9]{2}-[0-9]{4}-[0-9]{6}$"), "DIGITS_IN_PREFIX")
     .otherwise("OTHER")
)

# 3. Skirtingos blogos reikšmės ir jų dažniai
invalid_order_ids_distinct = order_id_check_df.filter(
    F.col("Order_ID_issue_type") != "VALID"
).groupBy("Order_ID_issue_type", "Order_ID_norm").count().orderBy(
    "Order_ID_issue_type", F.desc("count")
)

print("SKIRTINGOS BLOGOS Order_ID REIKŠMĖS")
invalid_order_ids_distinct.show(100, truncate=False)

# 4. Kiek distinct reikšmių kiekvienoje grupėje
invalid_distinct_summary = order_id_check_df.filter(
    F.col("Order_ID_issue_type") != "VALID"
).groupBy("Order_ID_issue_type").agg(
    F.count("*").alias("row_count"),
    F.countDistinct("Order_ID_norm").alias("distinct_bad_values")
).orderBy(F.desc("row_count"))

print("BLOGŲ Order_ID SUVESTINĖ")
invalid_distinct_summary.show(truncate=False)

# 5. Ar tie patys blogi Order_ID kartojasi per daug kartų
repeated_bad_ids = order_id_check_df.filter(
    F.col("Order_ID_issue_type") != "VALID"
).groupBy("Order_ID_norm").count().orderBy(F.desc("count"))

print("DAŽNIAUSIOS BLOGOS REIKŠMĖS")
repeated_bad_ids.show(20, truncate=False)

# 6. Papildoma patikra: ar blogi Order_ID susiję su kokiu nors konkrečiu klientu ar data
order_id_context = order_id_check_df.filter(
    F.col("Order_ID_issue_type") != "VALID"
).groupBy(
    "Order_ID_issue_type",
    "Order_ID_norm",
    "Customer_ID",
    "Order_Date"
).count().orderBy(F.desc("count"))

print("BLOGŲ Order_ID KONTEKSTAS")
order_id_context.show(50, truncate=False)

SKIRTINGOS BLOGOS Order_ID REIKŠMĖS
+-------------------+----------------+-----+
|Order_ID_issue_type|Order_ID_norm   |count|
+-------------------+----------------+-----+
|DIGITS_IN_PREFIX   |00-1234-123456  |704  |
|NULL               |NULL            |705  |
|TOO_LONG           |XY-12345-1234567|704  |
|TOO_SHORT          |AB-123-12345    |704  |
+-------------------+----------------+-----+

BLOGŲ Order_ID SUVESTINĖ
+-------------------+---------+-------------------+
|Order_ID_issue_type|row_count|distinct_bad_values|
+-------------------+---------+-------------------+
|NULL               |705      |0                  |
|TOO_SHORT          |704      |1                  |
|DIGITS_IN_PREFIX   |704      |1                  |
|TOO_LONG           |704      |1                  |
+-------------------+---------+-------------------+

DAŽNIAUSIOS BLOGOS REIKŠMĖS
+----------------+-----+
|Order_ID_norm   |count|
+----------------+-----+
|NULL            |705  |
|XY-12345-1234567|704  |
|AB-123-

In [13]:
from pyspark.sql import functions as F

# 1. Kiek blogų Order_ID prieš šalinimą
invalid_order_id_before = df.filter(
    F.col("Order_ID").isNull() |
    (~F.upper(F.trim(F.col("Order_ID"))).rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$"))
).count()

row_count_before = df.count()

print("PRIEŠ ŠALINIMĄ")
print("Viso eilučių:", row_count_before)
print("Blogų Order_ID eilučių:", invalid_order_id_before)

# 2. Pašalinamos eilutės su neatkuriamais Order_ID
df = df.filter(
    F.col("Order_ID").isNotNull() &
    F.upper(F.trim(F.col("Order_ID"))).rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$")
)

# 3. Patikra po šalinimo
row_count_after = df.count()

invalid_order_id_after = df.filter(
    F.col("Order_ID").isNull() |
    (~F.upper(F.trim(F.col("Order_ID"))).rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$"))
).count()

removed_rows = row_count_before - row_count_after

print("PO ŠALINIMO")
print("Viso eilučių:", row_count_after)
print("Blogų Order_ID eilučių:", invalid_order_id_after)
print("Pašalinta eilučių:", removed_rows)

PRIEŠ ŠALINIMĄ
Viso eilučių: 516445
Blogų Order_ID eilučių: 2817
PO ŠALINIMO
Viso eilučių: 513628
Blogų Order_ID eilučių: 0
Pašalinta eilučių: 2817


In [14]:
final_row_count = df.count()
print("Galutinis įrašų kiekis:", final_row_count)

Galutinis įrašų kiekis: 513628


In [15]:
allowed_segments = ["Consumer", "Corporate", "Home Office"]
allowed_payment_methods = ["Card", "Bank Transfer", "Cash", "PayPal"]
allowed_shipping_modes = ["Same Day", "First Class", "Second Class", "Standard Class"]
allowed_regions = ["East", "West", "North", "South", "Central"]

segment_invalid = df.filter(~F.col("Segment").isin(allowed_segments)).count()
payment_invalid = df.filter(~F.col("Payment_Method").isin(allowed_payment_methods)).count()
shipping_invalid = df.filter(~F.col("Shipping_Mode").isin(allowed_shipping_modes)).count()
origin_region_invalid = df.filter(~F.col("Origin_Region").isin(allowed_regions)).count()
destination_region_invalid = df.filter(~F.col("Destination_Region").isin(allowed_regions)).count()

print("Segment invalid:", segment_invalid)
print("Payment_Method invalid:", payment_invalid)
print("Shipping_Mode invalid:", shipping_invalid)
print("Origin_Region invalid:", origin_region_invalid)
print("Destination_Region invalid:", destination_region_invalid)

Segment invalid: 0
Payment_Method invalid: 0
Shipping_Mode invalid: 0
Origin_Region invalid: 0
Destination_Region invalid: 0


In [16]:
invalid_customer_id = df.filter(
    F.col("Customer_ID").isNull() |
    (~F.upper(F.trim(F.col("Customer_ID"))).rlike(r"^C-[0-9]{6}$"))
).count()

print("Netinkamų Customer_ID po galutinio valymo:", invalid_customer_id)

Netinkamų Customer_ID po galutinio valymo: 0


In [17]:
final_row_count = df.count()
final_duplicate_count = final_row_count - df.dropDuplicates().count()

print("Pilnų dublikatų po galutinio valymo:", final_duplicate_count)

Pilnų dublikatų po galutinio valymo: 0


In [18]:
null_exprs = []
for c, t in df.dtypes:
    if t == "string":
        null_exprs.append(
            F.sum(
                F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)
            ).alias(c)
        )
    else:
        null_exprs.append(
            F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        )

null_summary_final = df.select(null_exprs).collect()[0].asDict()

print("NULL / tuščios reikšmės po galutinio valymo:")
for col_name, bad_count in null_summary_final.items():
    if bad_count > 0:
        print(col_name, bad_count)

NULL / tuščios reikšmės po galutinio valymo:


In [19]:
ship_before_order_final = df.filter(
    F.col("Ship_Date") < F.col("Order_Date")
).count()

print("Ship_Date < Order_Date:", ship_before_order_final)

Ship_Date < Order_Date: 0


In [20]:
ship_before_order_final = df.filter(
    F.col("Ship_Date") < F.col("Order_Date")
).count()

print("Ship_Date < Order_Date:", ship_before_order_final)

Ship_Date < Order_Date: 0


In [21]:
df.printSchema()

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Order_Date: timestamp (nullable = true)
 |-- Ship_Date: timestamp (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Shipping_Mode: string (nullable = true)
 |-- Origin_City: string (nullable = true)
 |-- Origin_Region: string (nullable = true)
 |-- Origin_Latitude: double (nullable = true)
 |-- Origin_Longitude: double (nullable = true)
 |-- Destination_City: string (nullable = true)
 |-- Destination_Region: string (nullable = true)
 |-- Destination_Latitude: double (nullable = true)
 |-- Destination_Longitude: double (nullable = true)
 |-- Distance_km: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Weight_kg: double (nullable = true)
 |-- COGS: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Gross_Sales: double (nullable = true)
 |-- Net_Sales: double (nullable = true)
 |-- Profit: double (null

In [22]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import math

# Ensure Spark and df are defined
spark = SparkSession.builder.getOrCreate()
try:
    df.count()
except NameError:
    print("Variable 'df' not found. Loading from parquet...")
    df = spark.read.parquet("/content/sales_final.parquet")

print("=== 1. GALUTINIS EILUČIŲ KIEKIS ===")
print("Final row count:", df.count())

print("\n=== 2. SCHEMA ===")
df.printSchema()

# -----------------------------
# 3. NULL / empty / placeholder / NaN audit
# -----------------------------
print("\n=== 3. NULL / EMPTY / PLACEHOLDER / NaN AUDIT ===")

string_cols = [c for c, t in df.dtypes if t == "string"]
numeric_cols = [c for c, t in df.dtypes if t in ("double", "float", "int", "bigint", "long", "smallint", "tinyint")]

placeholder_values = ["", "-", "N/A", "NA", "NULL", "null", "None", "none"]

for c in string_cols:
    summary = df.select(
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias("nulls"),
        F.sum(F.when(F.trim(F.col(c)) == "", 1).otherwise(0)).alias("empty"),
        F.sum(F.when(F.trim(F.col(c)).isin(placeholder_values), 1).otherwise(0)).alias("placeholders")
    ).collect()[0]
    if summary["nulls"] > 0 or summary["empty"] > 0 or summary["placeholders"] > 0:
        print(f"{c}: nulls={summary['nulls']}, empty={summary['empty']}, placeholders={summary['placeholders']}")

for c in numeric_cols:
    nan_count = df.filter(F.isnan(F.col(c))).count() if dict(df.dtypes)[c] in ("double", "float") else 0
    null_count = df.filter(F.col(c).isNull()).count()
    if null_count > 0 or nan_count > 0:
        print(f"{c}: nulls={null_count}, NaN={nan_count}")

# -----------------------------
# 4. STRING NORMALIZATION AUDIT
# -----------------------------
print("\n=== 4. STRING NORMALIZATION AUDIT ===")

for c in string_cols:
    leading_or_trailing_spaces = df.filter(
        F.col(c).isNotNull() & (F.col(c) != F.trim(F.col(c)))
    ).count()

    multiple_internal_spaces = df.filter(
        F.col(c).isNotNull() & F.col(c).rlike(r".*\\s{2,}.*")
    ).count()

    if leading_or_trailing_spaces > 0 or multiple_internal_spaces > 0:
        print(f"{c}: leading/trailing={leading_or_trailing_spaces}, multi_spaces={multiple_internal_spaces}")

# papildomai case check kategoriniams laukams
categorical_cols = ["Segment", "Payment_Method", "Shipping_Mode", "Origin_City", "Origin_Region", "Destination_City", "Destination_Region"]

for c in categorical_cols:
    if c in df.columns:
        raw_distinct = df.select(c).distinct().count()
        trimmed_distinct = df.select(F.trim(F.col(c)).alias(c)).distinct().count()
        if raw_distinct != trimmed_distinct:
            print(f"{c}: raw_distinct={raw_distinct}, trimmed_distinct={trimmed_distinct}")

# -----------------------------
# 5. ALLOWED VALUES AUDIT
# -----------------------------
print("\n=== 5. ALLOWED VALUES AUDIT ===")

allowed_segments = ["Consumer", "Corporate", "Home Office"]
allowed_payment_methods = ["Card", "Bank Transfer", "Cash", "PayPal"]
allowed_shipping_modes = ["Same Day", "First Class", "Second Class", "Standard Class"]
allowed_regions = ["East", "West", "North", "South", "Central"]

checks = {
    "Segment": allowed_segments,
    "Payment_Method": allowed_payment_methods,
    "Shipping_Mode": allowed_shipping_modes,
    "Origin_Region": allowed_regions,
    "Destination_Region": allowed_regions,
}

for col_name, allowed in checks.items():
    bad = df.filter(~F.trim(F.col(col_name)).isin(allowed)).count()
    print(f"{col_name} invalid: {bad}")

# -----------------------------
# 7. EXTRA CROSS-COLUMN CHECKS
# -----------------------------
print("\n=== 7. EXTRA CROSS-COLUMN CHECKS ===")

print("Quantity outside [1,10]:", df.filter((F.col("Quantity") < 1) | (F.col("Quantity") > 10)).count())
print("Discount outside [0,0.3]:", df.filter((F.col("Discount") < 0) | (F.col("Discount") > 0.3)).count())
print("Distance_km <= 0:", df.filter(F.col("Distance_km") <= 0).count())
print("Weight_kg <= 0:", df.filter(F.col("Weight_kg") <= 0).count())
print("COGS <= 0:", df.filter(F.col("COGS") <= 0).count())
print("Ship_Date < Order_Date:", df.filter(F.col("Ship_Date") < F.col("Order_Date")).count())
print("Net_Sales > Gross_Sales:", df.filter(F.col("Net_Sales") > F.col("Gross_Sales")).count())
print("Profit mismatch:", df.filter(F.abs(F.col("Profit") - (F.col("Net_Sales") - F.col("COGS"))) > 0.01).count())

=== 1. GALUTINIS EILUČIŲ KIEKIS ===
Final row count: 513628

=== 2. SCHEMA ===
root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Order_Date: timestamp (nullable = true)
 |-- Ship_Date: timestamp (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Shipping_Mode: string (nullable = true)
 |-- Origin_City: string (nullable = true)
 |-- Origin_Region: string (nullable = true)
 |-- Origin_Latitude: double (nullable = true)
 |-- Origin_Longitude: double (nullable = true)
 |-- Destination_City: string (nullable = true)
 |-- Destination_Region: string (nullable = true)
 |-- Destination_Latitude: double (nullable = true)
 |-- Destination_Longitude: double (nullable = true)
 |-- Distance_km: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Weight_kg: double (nullable = true)
 |-- COGS: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Gross_Sales: double (null

In [23]:
from pyspark.sql import functions as F

string_cols = [c for c, t in df.dtypes if t == "string"]

print("=== MIXED VALUES CHECK ===")
for c in string_cols:
    pipe_count = df.filter(F.col(c).isNotNull() & F.col(c).contains("|")).count()
    colon_count = df.filter(F.col(c).isNotNull() & F.col(c).contains(":")).count()
    semicolon_count = df.filter(F.col(c).isNotNull() & F.col(c).contains(";")).count()
    slash_count = df.filter(F.col(c).isNotNull() & F.col(c).contains("/")).count()

    if pipe_count > 0 or colon_count > 0 or semicolon_count > 0 or slash_count > 0:
        print(f"{c}: |= {pipe_count}, := {colon_count}, ;= {semicolon_count}, /= {slash_count}")

=== MIXED VALUES CHECK ===


In [24]:
from pyspark.sql import functions as F

df_recheck = spark.read.parquet("/content/sales_final.parquet")

# 1. Gross_Sales / Net_Sales cleanup
df_recheck = df_recheck.withColumn("Gross_Sales", F.trim(F.col("Gross_Sales")))
df_recheck = df_recheck.withColumn("Gross_Sales", F.regexp_replace(F.col("Gross_Sales"), r"(?i)\s*eur\s*", ""))
df_recheck = df_recheck.withColumn("Gross_Sales", F.regexp_replace(F.col("Gross_Sales"), r"[€$]", ""))
df_recheck = df_recheck.withColumn("Gross_Sales", F.trim(F.col("Gross_Sales")))

df_recheck = df_recheck.withColumn("Net_Sales", F.upper(F.trim(F.col("Net_Sales"))))
df_recheck = df_recheck.withColumn("Net_Sales", F.regexp_replace(F.col("Net_Sales"), r"O", "0"))
df_recheck = df_recheck.withColumn("Net_Sales", F.regexp_replace(F.col("Net_Sales"), r"I", "1"))
df_recheck = df_recheck.withColumn("Net_Sales", F.regexp_replace(F.col("Net_Sales"), r"S", "5"))
df_recheck = df_recheck.withColumn("Net_Sales", F.trim(F.col("Net_Sales")))

# 2. cast
df_recheck = df_recheck.withColumn("Gross_Sales", F.col("Gross_Sales").cast("double"))
df_recheck = df_recheck.withColumn("Net_Sales", F.col("Net_Sales").cast("double"))

# 3. drop duplicates
df_recheck = df_recheck.dropDuplicates()

# 4. dates
df_recheck = df_recheck.withColumn("Order_Date", F.to_timestamp(F.col("Order_Date"), "yyyy-MM-dd"))
df_recheck = df_recheck.withColumn("Ship_Date", F.to_timestamp(F.col("Ship_Date"), "yyyy-MM-dd"))

# 5. Birštonas fix
df_recheck = df_recheck.withColumn(
    "Origin_Region",
    F.when(
        (F.col("Origin_City") == "Birštonas") &
        (F.round(F.col("Origin_Latitude"), 4) == 54.6054) &
        (F.round(F.col("Origin_Longitude"), 4) == 24.0296) &
        (F.col("Origin_Region") != "South"),
        F.lit("South")
    ).otherwise(F.col("Origin_Region"))
)

df_recheck = df_recheck.withColumn(
    "Destination_Region",
    F.when(
        (F.col("Destination_City") == "Birštonas") &
        (F.round(F.col("Destination_Latitude"), 4) == 54.6054) &
        (F.round(F.col("Destination_Longitude"), 4) == 24.0296) &
        (F.col("Destination_Region") != "South"),
        F.lit("South")
    ).otherwise(F.col("Destination_Region"))
)

print("Rows before Order_ID removal:", df_recheck.count())

Rows before Order_ID removal: 516445


In [25]:
bad_ids_df = df_recheck.filter(
    F.col("Order_ID").isNull() |
    (~F.upper(F.trim(F.col("Order_ID"))).rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$"))
)

print("Bad Order_ID rows:", bad_ids_df.count())
bad_ids_df.select("Order_ID").groupBy("Order_ID").count().show(truncate=False)

Bad Order_ID rows: 2817
+----------------+-----+
|Order_ID        |count|
+----------------+-----+
|NULL            |705  |
|XY-12345-1234567|704  |
|AB-123-12345    |704  |
|00-1234-123456  |704  |
+----------------+-----+



In [26]:
bad_ids_df.select(
    "Order_ID",
    F.length("Order_ID").alias("len"),
    F.hex(F.encode(F.col("Order_ID"), "UTF-8")).alias("hex_bytes")
).show(20, truncate=False)

+----------------+---+--------------------------------+
|Order_ID        |len|hex_bytes                       |
+----------------+---+--------------------------------+
|AB-123-12345    |12 |41422D3132332D3132333435        |
|AB-123-12345    |12 |41422D3132332D3132333435        |
|AB-123-12345    |12 |41422D3132332D3132333435        |
|00-1234-123456  |14 |30302D313233342D313233343536    |
|AB-123-12345    |12 |41422D3132332D3132333435        |
|XY-12345-1234567|16 |58592D31323334352D31323334353637|
|AB-123-12345    |12 |41422D3132332D3132333435        |
|XY-12345-1234567|16 |58592D31323334352D31323334353637|
|XY-12345-1234567|16 |58592D31323334352D31323334353637|
|AB-123-12345    |12 |41422D3132332D3132333435        |
|AB-123-12345    |12 |41422D3132332D3132333435        |
|00-1234-123456  |14 |30302D313233342D313233343536    |
|XY-12345-1234567|16 |58592D31323334352D31323334353637|
|XY-12345-1234567|16 |58592D31323334352D31323334353637|
|AB-123-12345    |12 |41422D3132332D3132333435  

In [27]:
bad_ids_df = bad_ids_df.withColumn("Order_ID_norm", F.upper(F.trim(F.col("Order_ID"))))

bad_ids_df.select(
    "Order_ID",
    "Order_ID_norm",
    F.length("Order_ID").alias("raw_len"),
    F.length("Order_ID_norm").alias("norm_len")
).show(20, truncate=False)

+----------------+----------------+-------+--------+
|Order_ID        |Order_ID_norm   |raw_len|norm_len|
+----------------+----------------+-------+--------+
|AB-123-12345    |AB-123-12345    |12     |12      |
|AB-123-12345    |AB-123-12345    |12     |12      |
|AB-123-12345    |AB-123-12345    |12     |12      |
|00-1234-123456  |00-1234-123456  |14     |14      |
|AB-123-12345    |AB-123-12345    |12     |12      |
|XY-12345-1234567|XY-12345-1234567|16     |16      |
|AB-123-12345    |AB-123-12345    |12     |12      |
|XY-12345-1234567|XY-12345-1234567|16     |16      |
|XY-12345-1234567|XY-12345-1234567|16     |16      |
|AB-123-12345    |AB-123-12345    |12     |12      |
|AB-123-12345    |AB-123-12345    |12     |12      |
|00-1234-123456  |00-1234-123456  |14     |14      |
|XY-12345-1234567|XY-12345-1234567|16     |16      |
|XY-12345-1234567|XY-12345-1234567|16     |16      |
|AB-123-12345    |AB-123-12345    |12     |12      |
|AB-123-12345    |AB-123-12345    |12     |12 

In [28]:
valid_ids_df = df_recheck.filter(
    F.col("Order_ID").isNotNull() &
    F.upper(F.trim(F.col("Order_ID"))).rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$")
)

# kiek skirtingų valid order_id turi tie klientai, kurie turi blogą order_id
bad_customer_valid_context = (
    bad_ids_df.select("Customer_ID").distinct()
    .join(valid_ids_df.select("Customer_ID", "Order_ID"), on="Customer_ID", how="left")
    .groupBy("Customer_ID")
    .agg(F.countDistinct("Order_ID").alias("valid_order_ids_for_customer"))
    .orderBy(F.desc("valid_order_ids_for_customer"))
)

bad_customer_valid_context.show(30, truncate=False)

+-----------+----------------------------+
|Customer_ID|valid_order_ids_for_customer|
+-----------+----------------------------+
|C-332687   |5                           |
|C-643115   |5                           |
|C-371129   |5                           |
|C-324997   |5                           |
|C-424697   |5                           |
|C-198327   |5                           |
|C-196751   |5                           |
|C-122850   |4                           |
|C-038881   |4                           |
|C-950544   |4                           |
|C-674669   |4                           |
|C-868288   |4                           |
|C-298115   |4                           |
|C-156612   |4                           |
|C-280136   |4                           |
|C-861922   |4                           |
|C-088564   |4                           |
|C-471372   |4                           |
|C-678445   |4                           |
|C-525315   |4                           |
|C-379267  

In [29]:
candidate_match = (
    bad_ids_df.select("Customer_ID", "Order_Date", "Order_ID")
    .join(
        valid_ids_df.select(
            F.col("Customer_ID").alias("v_Customer_ID"),
            F.col("Order_Date").alias("v_Order_Date"),
            F.col("Order_ID").alias("valid_Order_ID")
        ),
        (F.col("Customer_ID") == F.col("v_Customer_ID")) &
        (F.col("Order_Date") == F.col("v_Order_Date")),
        how="left"
    )
)

candidate_summary = candidate_match.groupBy("Customer_ID", "Order_Date", "Order_ID").agg(
    F.countDistinct("valid_Order_ID").alias("candidate_valid_ids")
).orderBy(F.desc("candidate_valid_ids"))

candidate_summary.show(50, truncate=False)

+-----------+-------------------+----------------+-------------------+
|Customer_ID|Order_Date         |Order_ID        |candidate_valid_ids|
+-----------+-------------------+----------------+-------------------+
|C-988735   |2025-08-25 00:00:00|AB-123-12345    |2                  |
|C-000206   |2025-09-28 00:00:00|00-1234-123456  |1                  |
|C-000246   |2026-12-31 00:00:00|XY-12345-1234567|1                  |
|C-000593   |2026-08-13 00:00:00|AB-123-12345    |1                  |
|C-000961   |2025-01-15 00:00:00|XY-12345-1234567|1                  |
|C-001194   |2025-07-17 00:00:00|XY-12345-1234567|1                  |
|C-002312   |2026-03-12 00:00:00|XY-12345-1234567|1                  |
|C-001237   |2025-11-28 00:00:00|XY-12345-1234567|1                  |
|C-002362   |2025-05-22 00:00:00|00-1234-123456  |1                  |
|C-001854   |2026-10-02 00:00:00|XY-12345-1234567|1                  |
|C-003391   |2025-03-20 00:00:00|AB-123-12345    |1                  |
|C-001

In [30]:
from pyspark.sql import functions as F

bad_ids_df = df_recheck.filter(
    F.col("Order_ID").isNull() |
    (~F.upper(F.trim(F.col("Order_ID"))).rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$"))
)

valid_ids_df = df_recheck.filter(
    F.col("Order_ID").isNotNull() &
    F.upper(F.trim(F.col("Order_ID"))).rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$")
)

join_cols_bad = [
    "Customer_ID", "Order_Date", "Ship_Date", "Segment", "Payment_Method",
    "Shipping_Mode", "Origin_City", "Origin_Region", "Origin_Latitude",
    "Origin_Longitude", "Destination_City", "Destination_Region",
    "Destination_Latitude", "Destination_Longitude", "Distance_km",
    "Quantity", "Weight_kg", "COGS", "Discount", "Gross_Sales", "Net_Sales", "Profit"
]

bad_alias = bad_ids_df.alias("b")
valid_alias = valid_ids_df.alias("v")

join_condition = None
for c in join_cols_bad:
    cond = F.col(f"b.{c}") == F.col(f"v.{c}")
    join_condition = cond if join_condition is None else (join_condition & cond)

exact_match_candidates = (
    bad_alias.join(valid_alias, join_condition, "left")
    .groupBy("b.Customer_ID", "b.Order_Date", "b.Order_ID")
    .agg(F.countDistinct("v.Order_ID").alias("exact_valid_matches"))
    .orderBy(F.desc("exact_valid_matches"))
)

exact_match_candidates.show(50, truncate=False)

exact_match_summary = exact_match_candidates.groupBy("exact_valid_matches").count().orderBy("exact_valid_matches")
exact_match_summary.show(truncate=False)

+-----------+-------------------+----------------+-------------------+
|Customer_ID|Order_Date         |Order_ID        |exact_valid_matches|
+-----------+-------------------+----------------+-------------------+
|C-398694   |2026-07-09 00:00:00|00-1234-123456  |1                  |
|C-647246   |2026-05-26 00:00:00|NULL            |1                  |
|C-048295   |2026-06-14 00:00:00|00-1234-123456  |1                  |
|C-868473   |2026-10-27 00:00:00|XY-12345-1234567|1                  |
|C-126327   |2025-02-04 00:00:00|AB-123-12345    |1                  |
|C-856175   |2025-04-05 00:00:00|AB-123-12345    |1                  |
|C-932371   |2026-01-01 00:00:00|XY-12345-1234567|1                  |
|C-001943   |2025-08-27 00:00:00|AB-123-12345    |1                  |
|C-460433   |2026-06-23 00:00:00|AB-123-12345    |1                  |
|C-682083   |2025-01-12 00:00:00|AB-123-12345    |1                  |
|C-487989   |2026-02-08 00:00:00|AB-123-12345    |1                  |
|C-442

In [31]:
from pyspark.sql import functions as F

# 1. Load fresh data to ensure we have the 'bad' rows
df_raw = spark.read.parquet('/content/sales_final.parquet')

# 2. Define what is valid and what is bad
valid_mask = F.upper(F.trim(F.col('Order_ID'))).rlike(r'^[A-Z]{2}-[0-9]{4}-[0-9]{6}$')

df_bad = df_raw.filter(~valid_mask | F.col('Order_ID').isNull())
df_valid = df_raw.filter(valid_mask)

# 3. Try to find 'Exact Attribute Matches'
# We join on every column EXCEPT Order_ID to see if a bad row matches a valid row exactly
match_cols = [c for c in df_raw.columns if c != 'Order_ID']

recovery_attempt = df_bad.alias('bad').join(
    df_valid.alias('valid'),
    on=match_cols,
    how='inner'
)

match_count = recovery_attempt.count()
print(f'Found {match_count} rows with bad IDs that have an identical valid counterpart.')

if match_count > 0:
    print('Sample of potentially recoverable IDs (Bad ID -> Suggested Valid ID):')
    recovery_attempt.select('bad.Order_ID', 'valid.Order_ID').show(10, truncate=False)
else:
    print('No exact matches found. The bad Order_ID rows appear to be unique or corrupt beyond simple recovery.')

Found 2778 rows with bad IDs that have an identical valid counterpart.
Sample of potentially recoverable IDs (Bad ID -> Suggested Valid ID):
+----------------+--------------+
|Order_ID        |Order_ID      |
+----------------+--------------+
|00-1234-123456  |YK-4277-378674|
|XY-12345-1234567|MT-9701-845419|
|AB-123-12345    |LV-5705-483787|
|00-1234-123456  |NH-1772-776183|
|00-1234-123456  |IX-3234-567649|
|00-1234-123456  |BF-4884-579316|
|NULL            |MM-3096-348446|
|XY-12345-1234567|EY-5511-210244|
|AB-123-12345    |IS-1977-657254|
|XY-12345-1234567|JH-3186-170061|
+----------------+--------------+
only showing top 10 rows


In [32]:
exact_match_summary = exact_match_candidates.groupBy("exact_valid_matches").count().orderBy("exact_valid_matches")
exact_match_summary.show(truncate=False)

+-------------------+-----+
|exact_valid_matches|count|
+-------------------+-----+
|0                  |28   |
|1                  |2789 |
+-------------------+-----+



In [33]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bad_ids_df = df_recheck.filter(
    F.col("Order_ID").isNull() |
    (~F.upper(F.trim(F.col("Order_ID"))).rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$"))
).withColumn("bad_row_id", F.monotonically_increasing_id())

valid_ids_df = df_recheck.filter(
    F.col("Order_ID").isNotNull() &
    F.upper(F.trim(F.col("Order_ID"))).rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$")
)

In [34]:
join_cols = [
    "Customer_ID", "Order_Date", "Ship_Date", "Segment", "Payment_Method",
    "Shipping_Mode", "Origin_City", "Origin_Region", "Origin_Latitude",
    "Origin_Longitude", "Destination_City", "Destination_Region",
    "Destination_Latitude", "Destination_Longitude", "Distance_km",
    "Quantity", "Weight_kg", "COGS", "Discount", "Gross_Sales", "Net_Sales", "Profit"
]

b = bad_ids_df.alias("b")
v = valid_ids_df.alias("v")

join_condition = None
for c in join_cols:
    cond = F.col(f"b.{c}") == F.col(f"v.{c}")
    join_condition = cond if join_condition is None else (join_condition & cond)

candidate_matches = b.join(v, join_condition, "left").select(
    F.col("b.bad_row_id"),
    F.col("b.Order_ID").alias("bad_Order_ID"),
    F.col("v.Order_ID").alias("candidate_valid_Order_ID")
)

In [35]:
candidate_count_per_bad_row = candidate_matches.groupBy("bad_row_id").agg(
    F.countDistinct("candidate_valid_Order_ID").alias("candidate_count")
)

candidate_count_per_bad_row.groupBy("candidate_count").count().orderBy("candidate_count").show(truncate=False)

+---------------+-----+
|candidate_count|count|
+---------------+-----+
|0              |28   |
|1              |2789 |
+---------------+-----+



In [36]:
from pyspark.sql import functions as F

print("=== Net_Sales >= COGS PATIKRA ===")

# Jei dirbi su originaliu df, kuriame Gross_Sales / Net_Sales dar string,
# pasidarom laikiną audit dataframe be overwrite
df_check = df

# Jei Net_Sales dar string, pabandom susitvarkyti tik auditui
if dict(df_check.dtypes)["Net_Sales"] == "string":
    df_check = df_check.withColumn("Net_Sales_check", F.upper(F.trim(F.col("Net_Sales"))))
    df_check = df_check.withColumn("Net_Sales_check", F.regexp_replace(F.col("Net_Sales_check"), r"O", "0"))
    df_check = df_check.withColumn("Net_Sales_check", F.regexp_replace(F.col("Net_Sales_check"), r"I", "1"))
    df_check = df_check.withColumn("Net_Sales_check", F.regexp_replace(F.col("Net_Sales_check"), r"S", "5"))
    df_check = df_check.withColumn("Net_Sales_check", F.col("Net_Sales_check").cast("double"))
else:
    df_check = df_check.withColumn("Net_Sales_check", F.col("Net_Sales"))

# Pati patikra
net_less_than_cogs = df_check.filter(
    F.col("Net_Sales_check").isNotNull() &
    F.col("COGS").isNotNull() &
    (F.col("Net_Sales_check") < F.col("COGS"))
).count()

net_equal_cogs = df_check.filter(
    F.col("Net_Sales_check").isNotNull() &
    F.col("COGS").isNotNull() &
    (F.col("Net_Sales_check") == F.col("COGS"))
).count()

net_greater_than_cogs = df_check.filter(
    F.col("Net_Sales_check").isNotNull() &
    F.col("COGS").isNotNull() &
    (F.col("Net_Sales_check") > F.col("COGS"))
).count()

print("Net_Sales < COGS:", net_less_than_cogs)
print("Net_Sales = COGS:", net_equal_cogs)
print("Net_Sales > COGS:", net_greater_than_cogs)

# Keli pavyzdžiai, jei yra pažeidimų
print("Pavyzdžiai, kur Net_Sales < COGS:")
df_check.filter(
    F.col("Net_Sales_check").isNotNull() &
    F.col("COGS").isNotNull() &
    (F.col("Net_Sales_check") < F.col("COGS"))
).select(
    "Order_ID", "Customer_ID", "Order_Date", "Gross_Sales", "Net_Sales", "Net_Sales_check", "Discount", "COGS", "Profit"
).show(20, truncate=False)

=== Net_Sales >= COGS PATIKRA ===
Net_Sales < COGS: 27954
Net_Sales = COGS: 0
Net_Sales > COGS: 485674
Pavyzdžiai, kur Net_Sales < COGS:
+--------------+-----------+-------------------+------------------+------------------+------------------+-------------------+------------------+--------------------+
|Order_ID      |Customer_ID|Order_Date         |Gross_Sales       |Net_Sales         |Net_Sales_check   |Discount           |COGS              |Profit              |
+--------------+-----------+-------------------+------------------+------------------+------------------+-------------------+------------------+--------------------+
|SL-6652-247868|C-004773   |2026-06-19 00:00:00|24.338081932122495|19.239393094544823|19.239393094544823|0.2094942753417307 |20.533721006386322|-1.2943279118414992 |
|BB-8074-371814|C-244765   |2025-08-05 00:00:00|13.480199532169241|10.181390908850913|10.181390908850913|0.24471511830711612|11.318204103124337|-1.136813194273424  |
|XZ-2429-845734|C-634710   |2025-

In [38]:
from pyspark.sql import functions as F

# 1. Galutinis eilučių kiekis
final_row_count = df.count()
print("GALUTINIS ĮRAŠŲ KIEKIS:", final_row_count)

# 2. Papildoma trumpa patikra, kad neliko blogų Order_ID
invalid_order_id_final = df.filter(
    F.col("Order_ID").isNull() |
    (~F.upper(F.trim(F.col("Order_ID"))).rlike(r"^[A-Z]{2}-[0-9]{4}-[0-9]{6}$"))
).count()

print("Netinkamų Order_ID po galutinio valymo:", invalid_order_id_final)

# 3. Išsaugom kaip vieną parquet failą naudojant coalesce(1)
output_path = "/content/sales_final_clean.parquet"
df.coalesce(1).write.mode("overwrite").parquet(output_path)

print("Parquet aplankas su vienu failu sukurtas čia:", output_path)

GALUTINIS ĮRAŠŲ KIEKIS: 513628
Netinkamų Order_ID po galutinio valymo: 0
Parquet aplankas su vienu failu sukurtas čia: /content/sales_final_clean.parquet


In [40]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Read the standalone file we just created
test_df = spark.read.parquet('/content/final_cleaned_sales.parquet')

print(f'--- FINAL AUDIT FOR: final_cleaned_sales.parquet ---')
print(f'Total Rows: {test_df.count()}')

# 2. Schema check
print('\n--- SCHEMA ---')
test_df.printSchema()

# 3. Comprehensive Null/Empty Check
print('\n--- NULL / EMPTY VALUES ---')
null_summary = test_df.select([F.sum(F.when(F.col(c).isNull() | (F.cast(F.col(c), 'string') == ''), 1).otherwise(0)).alias(c) for c in test_df.columns])
null_summary.show()

# 4. Business Logic Checks
print('\n--- BUSINESS LOGIC AUDIT ---')
logic_checks = {
    'Invalid Order_ID Format': test_df.filter(~F.col('Order_ID').rlike(r'^[A-Z]{2}-[0-9]{4}-[0-9]{6}$')).count(),
    'Invalid Customer_ID Format': test_df.filter(~F.col('Customer_ID').rlike(r'^C-[0-9]{6}$')).count(),
    'Quantity Errors (<1 or >10)': test_df.filter((F.col('Quantity') < 1) | (F.col('Quantity') > 10)).count(),
    'Discount Errors (<0 or >0.3)': test_df.filter((F.col('Discount') < 0) | (F.col('Discount') > 0.3)).count(),
    'Ship Date < Order Date': test_df.filter(F.col('Ship_Date') < F.col('Order_Date')).count(),
    'Net_Sales > Gross_Sales': test_df.filter(F.col('Net_Sales') > F.col('Gross_Sales')).count(),
    'Profit Mismatch (Net - COGS)': test_df.filter(F.abs(F.col('Profit') - (F.col('Net_Sales') - F.col('COGS'))) > 0.01).count(),
    'Full Row Duplicates': test_df.count() - test_df.dropDuplicates().count()
}

for check, count in logic_checks.items():
    print(f'{check}: {count}')

print('\n--- DATA SAMPLING ---')
test_df.select('Order_ID', 'Customer_ID', 'Order_Date', 'Net_Sales', 'Profit').show(5)

--- FINAL AUDIT FOR: final_cleaned_sales.parquet ---
Total Rows: 513628

--- SCHEMA ---
root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Order_Date: timestamp (nullable = true)
 |-- Ship_Date: timestamp (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Shipping_Mode: string (nullable = true)
 |-- Origin_City: string (nullable = true)
 |-- Origin_Region: string (nullable = true)
 |-- Origin_Latitude: double (nullable = true)
 |-- Origin_Longitude: double (nullable = true)
 |-- Destination_City: string (nullable = true)
 |-- Destination_Region: string (nullable = true)
 |-- Destination_Latitude: double (nullable = true)
 |-- Destination_Longitude: double (nullable = true)
 |-- Distance_km: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Weight_kg: double (nullable = true)
 |-- COGS: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Gross_Sales: dou

In [39]:
import os
import shutil

# Path where Spark saved the folder
folder_path = '/content/sales_final_clean.parquet'
final_file_name = '/content/final_cleaned_sales.parquet'

# Find the actual data file (the one starting with 'part-' and ending with '.parquet')
files = [f for f in os.listdir(folder_path) if f.startswith('part-') and f.endswith('.parquet')]

if files:
    source_file = os.path.join(folder_path, files[0])
    # Copy/Move it to the root content folder with a clean name
    shutil.copy(source_file, final_file_name)
    print(f'Successfully created single file: {final_file_name}')
else:
    print('No parquet part file found in the folder.')

Successfully created single file: /content/final_cleaned_sales.parquet
